# 15.3 `pytest` — Writing and Running Tests

**Prerequisites:** 15.1 Why Test and the assert Statement, 15.2 unittest, 07 Module and Packages  
**Target:** Python 3.12+ (notes flag 3.13/3.14 differences)

### What you'll learn
- Why `pytest` won: plain `assert`, with failure output that actually helps
- **Assertion rewriting** — how `assert a == b` produces a full diff
- Discovery rules, node IDs, and the exit codes CI reads
- `pytest.approx` for floats and `pytest.raises` for exceptions
- `@pytest.mark.parametrize` — real generated tests, not `subTest`
- The command line you will actually live in: `-q -v -x -k --lf --sw --tb`
- 🔴 Custom marks, and why an unregistered one can **fail your build**
- `pytest` runs your existing `unittest` tests unchanged

---

## What `pytest` adds

`unittest` (**15.2**) works. `pytest` is what almost everyone uses anyway, for four reasons:

| | `unittest` | `pytest` |
|---|---|---|
| A test is | a method on a `TestCase` subclass | **a plain function** |
| An assertion is | `self.assertEqual(a, b)` | **`assert a == b`** |
| A failure shows | `'a' != 'b'` | **both values, diffed, with the source line** |
| Table-driven tests | `subTest` — one test, sub-cases | **`@parametrize` — real separate tests** |
| Shared setup | `setUp` on the class | **fixtures, requested by name** (**15.4**) |

The second and third rows are the whole story. Being able to write `assert` — the statement
you already learned in **15.1** — and still get a rich failure report is the feature.

```bash
pip install pytest
```

Everything below runs `pytest` in a **temporary project directory** via `subprocess`, so you
see genuine output and nothing is written into these notes.

In [ ]:
import shutil
import subprocess
import sys
import tempfile
import textwrap
from pathlib import Path

WORK = Path(tempfile.mkdtemp(prefix="py153_"))


def make_project(files, name="proj"):
    """Create a temp project from {relative path: source} and return its path."""
    project = Path(tempfile.mkdtemp(prefix=f"{name}_", dir=WORK))
    for relpath, source in files.items():
        path = project / relpath
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(textwrap.dedent(source).lstrip("\n"), encoding="utf-8")
    return project


def pytest_in(project, *args, keep_cache=False):
    """Run pytest inside `project` and return its output, with the command echoed."""
    cmd = [sys.executable, "-m", "pytest", "--no-header"]
    if not keep_cache:                       # keeps .pytest_cache out of the way
        cmd += ["-p", "no:cacheprovider"]
    cmd += list(args)
    # 🔴 encoding="utf-8" matters: pytest prints characters like ± that the
    # Windows locale codec cannot represent, and they arrive as mojibake without it.
    done = subprocess.run(cmd, cwd=project, capture_output=True, text=True,
                          encoding="utf-8", errors="replace", timeout=300)
    banner = f"$ pytest {' '.join(args)}".rstrip()
    return (f"{banner}\n"
            f"{'-' * 70}\n"
            f"{(done.stdout + done.stderr).rstrip()}\n"
            f"{'-' * 70}\nexit code: {done.returncode}")


print("scratch:", WORK)
print("pytest :", subprocess.run([sys.executable, "-m", "pytest", "--version"],
                                 capture_output=True, text=True).stdout.strip())

## Discovery, node IDs and exit codes

`pytest` finds tests by convention. Defaults, all configurable:

| It looks for | Default rule |
|---|---|
| files | `test_*.py` or `*_test.py` |
| functions | `test*` |
| classes | `Test*` — 🔴 with **no `__init__` method** |
| methods in those classes | `test*` |

A **node ID** is how `pytest` names a single test. It is also how you re-run just that one:

```
tests/test_retry.py::TestBackoff::test_delay_is_capped
─────────┬───────── ─────┬────── ──────────┬─────────
       file          class (optional)     function
```

**Exit codes** — what your CI system reads:

| Code | Meaning |
|---|---|
| `0` | all tests passed |
| `1` | some tests failed |
| `2` | interrupted (collection error, `-x` after failure, Ctrl-C) |
| `3` | internal error |
| `4` | usage error (bad command line) |
| `5` | 🔴 **no tests were collected** — a green-looking build that tested nothing |

Code `5` is worth pinning up. A typo in a path, a renamed directory, a missing `test_` prefix
— and your pipeline reports "no failures" because it ran nothing.

In [ ]:
retry_module = """
    def retry_delay(attempt, base=1.0, ceiling=30.0):
        \"\"\"Seconds to wait before retry number `attempt` (0-based).\"\"\"
        if attempt < 0:
            raise ValueError(f"attempt must be >= 0, got {attempt}")
        return min(base * 2 ** attempt, ceiling)


    def cache_key(user_id, region):
        return f"user:{user_id}:{region}"
"""

first = make_project({
    "retry.py": retry_module,
    "test_retry.py": """
        from retry import retry_delay


        def test_first_attempt_waits_the_base_delay():
            assert retry_delay(0) == 1.0


        def test_delay_doubles_each_attempt():
            assert [retry_delay(n) for n in range(4)] == [1.0, 2.0, 4.0, 8.0]


        class TestCeiling:
            def test_delay_is_capped(self):
                assert retry_delay(9) == 30.0

            def test_base_above_ceiling_is_still_capped(self):
                assert retry_delay(0, base=100.0) == 30.0
    """,
}, name="first")

print(pytest_in(first, "-v"))

Four tests, no imports of a framework, no base class — and the node IDs on the
left are exactly what you paste back to re-run one.

Notice `TestCeiling` needed no `unittest.TestCase`. 🔴 But it must not define `__init__` —
`pytest` skips any `Test*` class with a constructor, silently, with only a warning.

### Proving exit code 5

In [ ]:
empty = make_project({
    "helpers.py": "def double(x):\n    return x * 2\n",
    "check_helpers.py": """
        from helpers import double


        def check_double():          # 🔴 named check_*, not test_* -> never collected
            assert double(2) == 4
    """,
}, name="empty")

print(pytest_in(empty, "-q"))
print("\n🔴 Zero tests ran, nothing failed. Only the exit code tells you.")
print("   In CI, add:  pytest --strict-markers -q || exit $?")
print("   and treat exit 5 as a build failure, not a pass.")

## Assertion rewriting — the headline feature

`assert a == b` is a bare Python statement. On failure it raises `AssertionError` with **no
information** — you saw that in **15.1**.

`pytest` gets around this by **rewriting the bytecode of your test modules at import time**.
It replaces each `assert` with code that captures the sub-expressions, so when the assertion
fails it can show you both sides.

```
     you write                pytest imports and rewrites it to (roughly)
     ─────────                ──────────────────────────────────────────
     assert f(x) == y   ──>   _left = f(x)
                              _right = y
                              if not (_left == _right):
                                  raise AssertionError(explain(_left, _right))
```

That is why `pytest` must **import** your test files rather than exec them, and why the trick
only works inside test modules (and `conftest.py`) — not in library code it merely imports.

Four failures below, of four different types. Read the diffs.

In [ ]:
diffs = make_project({
    "retry.py": retry_module,
    "test_diffs.py": """
        import pytest
        from retry import cache_key


        def test_string_diff():
            assert cache_key(7, "eu") == "user:7:us"


        def test_dict_diff():
            observed = {"state": "queued", "attempts": 1, "job": "build-42"}
            assert observed == {"state": "running", "attempts": 1, "job": "build-42"}


        def test_list_diff():
            assert ["queued", "running", "done"] == ["queued", "failed", "done"]


        def test_set_diff():
            assert {"queued", "running"} == {"queued", "done"}


        def test_bare_assert_has_nothing_to_show():
            assert cache_key(7, "eu").startswith("session:")
    """,
}, name="diffs")

print(pytest_in(diffs, "-q"))

Every one of those told you *which part* differed: the character position in the
string, the differing key in the dict, the index in the list, the missing set member.

🔴 The last test is the exception that proves the rule. `assert x.startswith("session:")` is a
**bare boolean** — there is nothing to diff, so `pytest` can only report `assert False`. When
that happens, either compare values instead:

```python
assert cache_key(7, "eu")[:8] == "session:"     # now there is a diff to show
```

or add a message — `assert cond, f"got {value!r}"` — which `pytest` prints.

> `-v` widens the diffs, `-vv` disables truncation entirely, and `--tb=short` / `--tb=line` /
> `--tb=no` control how much traceback comes with them.

## Floats: `pytest.approx`

The float problem from **2.2** and **15.2** again: `0.1 + 0.2 != 0.3`. `pytest.approx`
compares with a tolerance — by default **relative, 1e-6**.

```python
assert total == pytest.approx(0.3)                 # rel=1e-6 default
assert total == pytest.approx(0.3, rel=1e-3)       # within 0.1%
assert total == pytest.approx(0.3, abs=1e-9)       # absolute tolerance
assert residual == pytest.approx(0.0, abs=1e-6)    # 🔴 near zero — see below
```

It also works on lists, dicts and (if you have it) numpy arrays.

### 🔴 The zero case

The tolerance is `max(rel × expected, abs)`, with `rel=1e-6` and `abs=1e-12` by default. When
the expected value is **0**, the relative half contributes `1e-6 × 0 = 0` — so you silently
fall back on the **absolute** default:

| expected | effective tolerance |
|---|---|
| `approx(0.3)` | `3e-07` — comfortably loose |
| `approx(0.0)` | **`1e-12`** — a million times tighter |

So `approx(0)` is not "exact match only", but it *is* far stricter than you get anywhere else,
and a residual of `1e-11` — which looks like nothing — **fails**. Worse, passing `rel=` does
not help, because it is still multiplied by zero. Against an expected zero you must pass
`abs=` explicitly.

In [ ]:
approx_demo = make_project({
    "test_approx.py": """
        import pytest


        def test_exact_float_comparison_fails():
            assert 0.1 + 0.2 == 0.3


        def test_approx_is_the_fix():
            assert 0.1 + 0.2 == pytest.approx(0.3)


        def test_approx_works_on_containers():
            rates = [0.1 + 0.2, 1 / 3, 2 / 3]
            assert rates == pytest.approx([0.3, 0.333333, 0.666667], rel=1e-5)


        def test_approx_works_on_dicts():
            budget = {"cpu": 0.1 + 0.2, "mem": 0.7}
            assert budget == pytest.approx({"cpu": 0.3, "mem": 0.7})


        def test_the_effective_tolerances_differ_wildly():
            assert pytest.approx(0.3).tolerance == pytest.approx(3e-07)
            assert pytest.approx(0.0).tolerance == 1e-12


        def test_a_tiny_residual_still_fails_against_zero():
            residual = 1e-11                    # looks like nothing...
            assert residual == pytest.approx(0.0)          # 🔴 ...but this fails


        def test_rel_does_not_help_against_zero():
            residual = 1e-11
            assert residual == pytest.approx(0.0, rel=1e-3)  # 🔴 rel x 0 == 0, still fails


        def test_abs_is_what_you_need():
            residual = 1e-11
            assert residual == pytest.approx(0.0, abs=1e-6)
    """,
}, name="approx")

print(pytest_in(approx_demo, "-q", "--tb=line"))

Read the summary line: **3 failed, 5 passed**. The two failures against zero are
the point — `1e-11` is not "close enough to zero" by default, and adding `rel=1e-3` does not
change that by a single digit, because it is multiplied by the expected value of `0`.

## Exceptions: `pytest.raises`

```python
with pytest.raises(ValueError):                     # must raise ValueError
    retry_delay(-1)

with pytest.raises(ValueError, match=r"must be >= 0"):   # ...and the message must match
    retry_delay(-1)

with pytest.raises(ValueError) as excinfo:          # capture it for inspection
    retry_delay(-1)
assert "got -1" in str(excinfo.value)
```

🔴 **`match` takes a regular expression and uses `re.search`** (**09**), not a plain substring
and not `re.fullmatch`. Characters like `(`, `)`, `[`, `.`, `$` and `+` are *pattern* syntax —
`match="cost is $5 (approx)"` will not do what you expect. Wrap literal text in `re.escape()`.

🔴 **Never write `pytest.raises(Exception)`.** It passes for a `TypeError` from a typo in your
own test, so the test can pass while testing nothing.

In [ ]:
raises_demo = make_project({
    "retry.py": retry_module,
    "test_raises.py": """
        import re
        import pytest
        from retry import retry_delay


        def test_negative_attempt_is_rejected():
            with pytest.raises(ValueError):
                retry_delay(-1)


        def test_the_message_explains_what_was_wrong():
            with pytest.raises(ValueError, match=r"attempt must be >= 0"):
                retry_delay(-1)


        def test_inspecting_the_exception_object():
            with pytest.raises(ValueError) as excinfo:
                retry_delay(-5)
            assert "got -5" in str(excinfo.value)
            assert excinfo.type is ValueError


        def test_match_is_a_regex_not_a_substring():
            # The real message contains ">= 0", and `>` is fine, but if the message
            # had parentheses this pattern would silently mean something else.
            with pytest.raises(ValueError, match=re.escape("attempt must be >= 0, got -1")):
                retry_delay(-1)


        def test_too_loose_passes_for_the_wrong_reason():
            with pytest.raises(Exception):        # 🔴 catches the typo below, not the bug
                retry_delay(-1, bogus_kwarg=True)


        def test_a_missing_exception_is_a_failure():
            with pytest.raises(ValueError):
                retry_delay(3)                    # does not raise at all
    """,
}, name="raises")

print(pytest_in(raises_demo, "-q"))

Look at `test_too_loose_passes_for_the_wrong_reason`: it **passed**. But
`retry_delay` never raised a `ValueError` there — it raised `TypeError` because
`bogus_kwarg` is not a parameter. `pytest.raises(Exception)` swallowed the difference and
reported success. Always name the exact exception class.

And `test_a_missing_exception_is_a_failure` shows the other half: if the block does **not**
raise, `pytest.raises` fails with `DID NOT RAISE`. That is what makes it a real assertion
rather than a `try`/`except` that quietly passes.

## `@pytest.mark.parametrize`

`subTest` (**15.2**) reported every case but still counted as **one** test. `parametrize`
generates **real, separate tests** — separately named, separately counted, individually
re-runnable by node ID.

```
@pytest.mark.parametrize("attempt,expected", [(0, 1.0), (1, 2.0), (9, 30.0)])
                          ───────┬────────    ───────────┬──────────────────
                          parameter names        one tuple per generated test
def test_backoff(attempt, expected):
    assert retry_delay(attempt) == expected
```

Stacking two `parametrize` decorators gives you the **cartesian product** — 3 × 2 = 6 tests.

In [ ]:
param_demo = make_project({
    "retry.py": retry_module,
    "test_param.py": """
        import pytest
        from retry import retry_delay, cache_key


        @pytest.mark.parametrize("attempt,expected", [
            (0, 1.0),
            (1, 2.0),
            (2, 4.0),
            (5, 30.0),
            (9, 30.0),
        ])
        def test_backoff_schedule(attempt, expected):
            assert retry_delay(attempt) == expected


        @pytest.mark.parametrize("attempt,expected", [
            pytest.param(0, 1.0, id="first-attempt"),
            pytest.param(9, 30.0, id="capped-at-ceiling"),
            pytest.param(-1, 0.0, id="negative-should-raise",
                         marks=pytest.mark.xfail(raises=ValueError)),
        ])
        def test_backoff_with_readable_ids(attempt, expected):
            assert retry_delay(attempt) == expected


        @pytest.mark.parametrize("region", ["eu", "us", "ap"])
        @pytest.mark.parametrize("user_id", [1, 42])
        def test_cache_key_shape(user_id, region):
            assert cache_key(user_id, region) == f"user:{user_id}:{region}"
    """,
}, name="param")

print(pytest_in(param_demo, "-v"))

Three things in that output:

- **Auto-generated IDs** — `test_backoff_schedule[0-1.0]`, built from the values.
- **`pytest.param(..., id="capped-at-ceiling")`** replaces them with something readable. Worth
  doing whenever the values are not self-explanatory.
- **`marks=pytest.mark.xfail(raises=ValueError)`** marks *one case* as an expected failure —
  the `parametrize`-level equivalent of `@unittest.expectedFailure`. It reports `xfail`, and
  if it ever *passes* you get `XPASS`.
- The stacked decorators produced **6** tests: 2 user IDs × 3 regions.

🔴 The order reads bottom-up: the decorator **closest to the function** varies fastest.

## The same test, both ways

Nothing here is a rewrite: this is one behaviour expressed twice, run in one command.

In [ ]:
both = make_project({
    "retry.py": retry_module,
    "test_style_unittest.py": """
        import unittest
        from retry import retry_delay


        class BackoffTests(unittest.TestCase):
            def test_delay_is_capped_at_the_ceiling(self):
                self.assertEqual(retry_delay(9), 30.0)

            def test_negative_attempt_is_rejected(self):
                with self.assertRaises(ValueError):
                    retry_delay(-1)

            def test_schedule(self):
                for attempt, expected in [(0, 1.0), (1, 2.0), (2, 4.0)]:
                    with self.subTest(attempt=attempt):
                        self.assertEqual(retry_delay(attempt), expected)
    """,
    "test_style_pytest.py": """
        import pytest
        from retry import retry_delay


        def test_delay_is_capped_at_the_ceiling():
            assert retry_delay(9) == 30.0


        def test_negative_attempt_is_rejected():
            with pytest.raises(ValueError):
                retry_delay(-1)


        @pytest.mark.parametrize("attempt,expected", [(0, 1.0), (1, 2.0), (2, 4.0)])
        def test_schedule(attempt, expected):
            assert retry_delay(attempt) == expected
    """,
}, name="both")

print(pytest_in(both, "-v"))

Both files ran under one `pytest` command. Two differences worth noting in
that output:

- The `unittest` file contributes **3** tests; the `pytest` file contributes **5**, because
  `parametrize` generated three real tests where `subTest` produced one.
- 🔴 **Fixtures do not work on `unittest.TestCase` subclasses.** `pytest` runs them, but they
  cannot request fixtures as arguments (**15.4**). That is the one real limit of the
  "adopt `pytest`, keep your tests" story.

## The command line

You will use six flags for the rest of your career:

| Flag | Does |
|---|---|
| `-q` / `-qq` | quieter output — the default for a run you expect to pass |
| `-v` / `-vv` | one line per test / no diff truncation |
| `-x` | **stop at the first failure** (`--maxfail=1`) |
| `-k EXPR` | select by substring/boolean over names: `-k "ceiling and not slow"` |
| `-m EXPR` | select by **mark**: `-m "not slow"` |
| `--lf` | **last failed** — rerun only the tests that failed last time |
| `--ff` | failed first, then the rest |
| `--sw` | stepwise: stop at the first failure, resume there next time |
| `--tb=short\|line\|no` | shrink tracebacks |
| `-ra` | a summary of all non-passing tests at the end |
| `--collect-only` | show what *would* run, and run nothing |
| `--durations=10` | the 10 slowest tests — how you find the ones to fix |

The workflow that these support: run `-q`, get failures, `--lf -x` to work through them one at
a time, then `-q` again to confirm.

In [ ]:
cli = make_project({
    "retry.py": retry_module,
    "test_cli.py": """
        import pytest
        from retry import retry_delay, cache_key


        @pytest.mark.parametrize("attempt,expected", [(0, 1.0), (1, 2.0), (2, 4.0)])
        def test_backoff(attempt, expected):
            assert retry_delay(attempt) == expected


        def test_ceiling_is_respected():
            assert retry_delay(9) == 30.0


        def test_cache_key_is_broken_on_purpose():
            assert cache_key(7, "eu") == "user:7:us"


        def test_another_one_that_passes():
            assert cache_key(1, "ap") == "user:1:ap"
    """,
}, name="cli")

print(pytest_in(cli, "--collect-only", "-q"))
print()
print(pytest_in(cli, "-k", "backoff or ceiling", "-v"))

`--collect-only` listed the six node IDs without running anything — the fastest
way to answer "is my test actually being picked up?".

`-k "backoff or ceiling"` selected 4 of 6 and **deselected** the rest. `-k` matches against
the whole node ID, so it can select by file, class, function or parameter ID.

Now the failure workflow. Note `--lf` needs the cache, so this pair of runs keeps it.

In [ ]:
print(pytest_in(cli, "-x", "-q", "--tb=line", keep_cache=True))
print()
print("Now only the failures, from the cache pytest just wrote:")
print()
print(pytest_in(cli, "--lf", "-q", "--tb=no", keep_cache=True))

## Marks, and 🔴 the one that can break your build

A **mark** is a label on a test. Built-in ones:

| Mark | Meaning |
|---|---|
| `@pytest.mark.skip(reason=...)` | never run |
| `@pytest.mark.skipif(cond, reason=...)` | conditional skip |
| `@pytest.mark.xfail` | expected to fail; `XPASS` if it doesn't |
| `@pytest.mark.xfail(raises=ValueError, strict=True)` | must fail, *this* way |

You can also invent your own — `@pytest.mark.slow`, `@pytest.mark.integration` — and select
with `-m "not slow"`.

### 🔴 The trap

An **unregistered** mark is only a `PytestUnknownMarkWarning`. Two things make that dangerous:

1. A typo — `@pytest.mark.slwo` — silently does nothing. `-m "not slow"` still runs it.
2. Under **`-W error`**, or `filterwarnings = error` in your config (a common and otherwise
   good setting), that warning becomes a **collection error**: exit code 2, zero tests run.

The next cell demonstrates all three states: unregistered, unregistered under `-W error`, and
registered properly.

In [ ]:
marked = {
    "retry.py": retry_module,
    "test_marks.py": """
        import pytest
        from retry import retry_delay


        def test_fast_one():
            assert retry_delay(0) == 1.0


        @pytest.mark.slow
        def test_slow_one():
            assert retry_delay(9) == 30.0


        @pytest.mark.slwo                     # 🔴 typo - silently not the same mark
        def test_typo_mark():
            assert retry_delay(1) == 2.0
    """,
}

unregistered = make_project(marked, name="marks_unreg")

print("### 1. unregistered: just a warning, and `-m 'not slow'` still runs the typo'd test")
print(pytest_in(unregistered, "-q", "-m", "not slow"))

In [ ]:
print("### 2. the same project under -W error -- the warning is now fatal")
print(pytest_in(unregistered, "-q", "-W", "error"))

In [ ]:
registered = make_project({
    **marked,
    "pytest.ini": """
        [pytest]
        markers =
            slow: takes more than a second; excluded from the pre-commit run
        filterwarnings =
            error
    """,
}, name="marks_reg")

print("### 3. registered in pytest.ini, and run with --strict-markers")
print(pytest_in(registered, "-q", "--strict-markers"))

Three different outcomes from the same test file:

| Run | Result |
|---|---|
| unregistered, `-m "not slow"` | passes, **but the typo'd test ran anyway** — the mark did nothing |
| unregistered, `-W error` | **exit 2, collection error, zero tests run** |
| registered + `--strict-markers` | the typo is caught as a hard error, naming the mark |

🔴 **Always use `--strict-markers`** (or `addopts = --strict-markers` in your config). It
turns "unknown mark" from a warning you scroll past into an error that names the typo. This is
the same lesson as `-W error::SyntaxWarning` from **15.1**: promote the warnings you cannot
afford to miss.

Configuration lives in `pytest.ini`, `pyproject.toml` (`[tool.pytest.ini_options]`), `tox.ini`
or `setup.cfg` — and **the file that holds it defines the `rootdir`**, which is how `pytest`
resolves relative paths and imports. That, and `conftest.py`, are **15.4**; packaging and
`pyproject.toml` proper are **18**.

In [ ]:
# ---- tidy up ----
shutil.rmtree(WORK, ignore_errors=True)
print("scratch removed:", not WORK.exists())

---

## Common Mistakes & Pitfalls

1. 🔴 **`pytest.raises(Exception)`.** It passes when your own test has a typo. Name the exact exception class, and add `match=` so a different error of the same type cannot sneak through.
2. 🔴 **Forgetting `match=` is a regex.** `match="cost (approx)"` is a group, not literal text. Wrap literal strings in `re.escape()` (**09**).
3. 🔴 **Unregistered marks.** A typo'd mark silently does nothing, and under `filterwarnings = error` it aborts collection entirely. Register them and use `--strict-markers`.
4. **Treating exit code 5 as success.** No tests collected is not a passing build.
5. **A `Test*` class with an `__init__`.** `pytest` skips it with only a warning, so the whole class silently never runs.
6. **Comparing floats with `==`.** Use `pytest.approx` — and remember `rel` cannot match zero, so pass `abs` when the expected value is 0.
7. **Bare boolean asserts** like `assert x.startswith(...)`. There is nothing to diff, so the failure says `assert False`. Compare values, or add a message.
8. **Expecting fixtures to work on `unittest.TestCase` subclasses.** They do not — `pytest` runs those tests, but cannot inject fixtures into them.
9. **Leaving `-x` in your CI command.** You then fix one failure per CI run instead of seeing all of them at once. `-x` is for local work.

## Best Practices

- Write tests as plain functions with plain `assert`; reach for a class only to group related tests.
- Put `addopts = -ra --strict-markers` and `filterwarnings = error` in your config from day one — retrofitting them to a large suite is painful.
- Use `@parametrize` instead of loops, and give cases readable IDs with `pytest.param(..., id=...)`.
- Use `-k` and node IDs to run one test while debugging; `--lf -x` to work through a list of failures.
- Prefer `xfail(strict=True)` over `skip` for known bugs, so the suite tells you when one is fixed.
- Run `--collect-only` whenever you are unsure a test is being picked up.
- Check `--durations=10` occasionally; a suite people will not run is a suite that does not protect anything.
- Keep `pytest` and your source importable the same way in CI and locally — most 'works on my machine' test failures are import path problems (**18**).

## Practice Exercises

Try these before moving on.

1. Write `test_retry.py` for `retry_delay` using `@parametrize` for the schedule, `pytest.raises` for the negative attempt, and `pytest.approx` for a jittered variant.
2. Take the `test_bare_assert_has_nothing_to_show` example and rewrite it two ways so the failure output is actually useful. Which do you prefer, and why?
3. 🔴 Write a test that passes for the wrong reason using `pytest.raises(Exception)`, then tighten it until it genuinely tests what you meant.
4. Create a project with a `pytest.ini` registering `slow` and `integration` marks. Verify that `-m "not slow"` excludes the right tests, and that a typo'd mark is now an error.
5. Run a suite with `--collect-only -q` and count the tests. Now rename one file from `test_x.py` to `x_test.py` and confirm it is still collected. Rename it to `checks_x.py` and confirm the exit code becomes 5.
6. Convert `15.2`'s `JobStateTests` to pytest functions. Which assertions get shorter? Which `assert*` methods have no plain-`assert` equivalent, and what do you use instead?
7. **Interview question:** how does `pytest` make `assert a == b` print a diff, when the `assert` statement itself carries no such information? What are the limits of that mechanism?

---

## Version notes

| Version | Change |
|---|---|
| **pytest 9** | `--strict-markers` behaviour and the unknown-mark warning are unchanged, but many projects now ship `filterwarnings = error`, which turns that warning into the collection error shown above |
| **pytest 8** | nose-style `setup()` / `teardown()` free functions **removed**; use fixtures or `setup_method` |
| **pytest 8** | `--import-mode=importlib` became fully supported for `src/` layouts without `__init__.py` |
| **pytest 7** | `pytest.approx` gained dict support; `tmp_path` fully replaced `tmpdir` (**15.4**) |
| Python 3.12 | `unittest` alias removals (**15.2**) apply to `unittest`-style tests run under `pytest` too |

This notebook was written against **pytest 9.1** on **Python 3.14**.

## Where next

| Notebook | Covers |
|---|---|
| **15.4 Fixtures, Isolation and Test Data** | `conftest.py`, scopes, `tmp_path`, `monkeypatch`, `capsys` |
| **15.5 Test Doubles** | `unittest.mock`, patching, and where to patch |
| **15.6 Testing in Practice** | coverage, property-based testing, layout, CI |

## Related

- **15.1** — `assert`, and the hand-rolled runner this replaces
- **15.2** — `unittest`, which `pytest` runs unchanged
- **09 Regular Expression** — the regex `match=` expects
- **2.2 Numeric Datatype** — why `approx` exists
- **07 Module and Packages** — imports, which is what `rootdir` problems really are
- **18 Tooling, Packaging and Environments** — `pyproject.toml` configuration